In [ ]:
import jax

# Un/comment this for double/single precision:
# jax.config.update('jax_enable_x64', True)

import copy
import equinox as eqx
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import UncertainSCI._equinox as _eqx
import UncertainSCI.gp as gp


D = 1  
C = 1

np.random.seed(0x01234567)
seed = 0xdeadbeef  # Used in gp.GaussianProcess initialization.

FIGSIZE = (7, 7 / 1.6)
FIGDPI = None

def print_loss_here(g):  # Simple helper for this notebook; does not generalize.
    print(
        f'Loss at D = {jnp.squeeze(g.k.D()):.4e} (length scale {1 / jnp.sqrt(jnp.squeeze(g.k.D())):.4e}): '
        f'{g.loss_hyperparameters(train_x, train_y, train_s):.4e}'
    )


In [ ]:
x_plot = jnp.linspace(0, 2 * jnp.pi, 1000).reshape((-1, D))

def f_hidden(x):
    return 2 * jnp.cos(x) * jnp.sin(4 * x)

def f_noisy(x):
    y = f_hidden(x)
    return (
        y + 1e-3 * np.random.randn(*y.shape),
        1e-3 * jnp.ones_like(y)
    )


In [ ]:
mu = gp.mean.Affine(
    dim=D,
    cdim=C,
    a=0. * jnp.ones((C, D)),
    b=0.,
    a_is_static=True,
    b_is_static=True
)
k = gp.kernel.Gaussian(
    dim=D,
    cdim=C,
    D=jnp.ones((1, 1)),
)
g = gp.GaussianProcess(dim=D, cdim=C, mu=mu, k=k, seed=seed, nugget=1e-3)
# g = gp.GaussianProcess(dim=D, cdim=C, mu=mu, k=k, seed=seed, nugget=1e-4)


In [ ]:


def plot_in_subplots(nrows=1, ncols=1, **kwargs):
    """
    Returns ``(fig, axes)`` with plot scaled correctly for ``(nrows, ncols)``.
    """
    if 'figsize' in kwargs:
        raise ValueError("kwargs had 'figsize' key: that's the whole point of this function!")
    return plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * FIGSIZE[0], nrows * FIGSIZE[1]),
        **kwargs
    )


In [ ]:
fig, (ax1, ax2) = plot_in_subplots(2, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'prior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'prior')
plt.show()


In [ ]:
N_INIT = 10

train_x = jnp.linspace(0, 2 * jnp.pi, N_INIT).reshape(-1, D)
train_y, train_s = f_noisy(train_x)    

g.condition(train_x, train_y, train_s)


Plot what that looks like:

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior', colorlast=False)
gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


Tune:

In [ ]:
losses = g.tune()

plt.figure(figsize=FIGSIZE)
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()


Plot what things look like after tuning:

In [ ]:
fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior', colorlast=False)
gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
plt.show()

print_loss_here(g)


Iteratively sample points to choose for tuning, then tune:

In [ ]:
N_TRAIN = 10
N_SAMPLE = 20

x_sample = jnp.linspace(0, 2 * jnp.pi, N_SAMPLE).reshape(-1, D)

for i in range(N_TRAIN):
    # Greedily sample posterior variance:
    _v = jnp.diag(g.posterior_covariance(x_sample, x_sample))
    _x = x_sample[jnp.argmax(_v)].reshape(-1, D)

    # Iteratively optimize posterior variance:
    _x = g.get_sample_point(_x.reshape((1,)), ranges=[[0.], [2 * jnp.pi]])

    # Observe that point:
    _y, _s = f_noisy(_x)

    train_x = jnp.concat((train_x, _x))
    train_y = jnp.concat((train_y, _y))
    train_s = jnp.concat((train_s, _s))

    # Condition with new data:
    g.condition(train_x, train_y, train_s)
    g.tune()

    # Plot:
    fig, (ax1, ax2, ax3) = plot_in_subplots(3, 1, dpi=FIGDPI)
    gp.vis.plot_distribution_mean(ax1, g, x_plot, f_hidden, 'posterior')
    gp.vis.plot_distribution_variance(ax2, g, x_plot, 'posterior')
    gp.vis.plot_loss_landscape(ax3, g, ('k', 'D'), jnp.logspace(-2, 2, 1000))
    plt.show()
    print_loss_here(g)
    print('\n' * 3)
